# 04.3 `id()`, `type()`, and Object Identity

Every object carries three fixed facts. Two of them never change for that
object's entire life. This notebook shows how to inspect them, and — more
importantly — why `is` sometimes gives answers that look wrong.

## Theory

### The three properties, revisited

<table>
<tr><th>Property</th><th>Function</th><th>Changes?</th></tr>
<tr><td>Identity</td><td><code>id(x)</code></td><td>Never</td></tr>
<tr><td>Type</td><td><code>type(x)</code></td><td>Never</td></tr>
<tr><td>Value</td><td><code>x</code></td><td>Only if mutable</td></tr>
</table>

### What `id()` actually returns

In CPython, `id(x)` returns the object's **memory address**. This is an
implementation detail — the language only guarantees that the number is unique
and constant for the object's lifetime.

Two consequences:

1. The numbers differ on every run. Never store or compare them across runs.
2. An id can be **reused** after an object is freed. A new object may receive the
   id of a dead one.

That second point causes genuinely confusing results, demonstrated below.

### `is` is just an id comparison

```python
a is b        # exactly equivalent to:  id(a) == id(b)
```

There is no magic. `is` cannot be overridden by a class, which is precisely why
it is the correct way to test for `None`.

### Interning: why `is` sometimes surprises you

CPython **caches** some immutable objects so that equal values share one object:

- **Small integers** from −5 to 256 are pre-created at startup
- **Short string literals** that look like identifiers are interned at compile
  time

This is an optimisation, not a language rule. It means `a is b` can be `True`
for two separately written literals — and then `False` for slightly different
ones. Relying on it is a bug.

In [ ]:
# The three properties of one object.
values = [10, 20, 30]

print("VALUE   :", values)
print("TYPE    :", type(values))
print("TYPE name:", type(values).__name__)
print("IDENTITY:", id(values))

# Mutating changes the value but NOT the identity or type.
identity_before = id(values)
values.append(40)

print("")
print("After append(40):")
print("   value   :", values, "<- changed")
print("   type    :", type(values).__name__, "<- same")
print("   identity:", id(values) == identity_before, "same id? <- same object")

# Rebinding creates a different object entirely.
values = [99]
print("")
print("After values = [99]:")
print("   same identity as before?", id(values) == identity_before)
print("   this is a NEW object - the name was repointed")

## `is` versus `==`, precisely

`is` compares identity. `==` compares value, and a class can define what that
means.

In [ ]:
# Equal values, separate objects.
list_one = [1, 2, 3]
list_two = [1, 2, 3]

print("list_one == list_two:", list_one == list_two)
print("list_one is list_two:", list_one is list_two)
print("id(list_one) == id(list_two):", id(list_one) == id(list_two))
print("")
print("`is` is literally an id comparison - nothing more.")

# A class can make == mean whatever it likes. It cannot touch `is`.
class AlwaysEqual:
    """A class that claims equality with everything."""

    def __eq__(self, other):
        # Deliberately absurd, to prove the point.
        return True


first = AlwaysEqual()
second = AlwaysEqual()

print("")
print("With a custom __eq__:")
print("   first == second:", first == second, "<- the class decided this")
print("   first is second:", first is second, "<- identity cannot be faked")

print("")
print("This is why `x is None` is correct and `x == None` is fragile:")
print("a class could define __eq__ to return True for None.")

## Small integer caching

CPython pre-creates the integers from −5 to 256 when it starts. Every use of
those values shares one object.

In [ ]:
# Inside the cached range, separate assignments share an object.
small_a = 100
small_b = 100

print("small integers (100):")
print("   same object?", small_a is small_b)
print("   id a:", id(small_a))
print("   id b:", id(small_b))

# Outside the range, you usually get distinct objects.
# We build them at runtime so the compiler cannot fold them into one constant.
large_a = int("1000")
large_b = int("1000")

print("")
print("larger integers (1000, built at runtime):")
print("   equal?      ", large_a == large_b)
print("   same object?", large_a is large_b)

# Find where the cache ends.
print("")
print("Testing the cache boundary:")
for candidate in [255, 256, 257, 258]:
    # Build both at runtime to avoid compile-time constant folding.
    first = int(str(candidate))
    second = int(str(candidate))
    print(f"   {candidate}: same object? {first is second}")

print("")
print("The cache covers -5 to 256. This is a CPython optimisation,")
print("not a language guarantee - never write code that depends on it.")

### Why the boundary matters

This is the classic interview question, and more importantly a real source of
bugs when people use `is` to compare numbers.

In [ ]:
# The bug: using `is` to compare values instead of ==.
# We build the values at runtime so Python cannot fold them into
# one shared constant, which is what makes the difference visible.
cached_value = int("100")
large_value = int("1000")

print("Comparing with == (correct):")
print("   int('100')  == 100 :", cached_value == 100)
print("   int('1000') == 1000:", large_value == 1000)

print("")
print("Comparing with `is` (wrong, and inconsistent):")
print("   int('100')  is 100 :", cached_value is 100)
print("   int('1000') is 1000:", large_value is 1000)

print("")
print("The first is True only because 100 is in the cache.")
print("The second is False. Same code shape, different answer.")
print("")
print("Python even warns you about this - writing `x is 100` in a script")
print("raises a SyntaxWarning telling you to use ==.")

## String interning

Python interns strings that look like identifiers, at compile time. Strings built
at runtime are usually separate objects.

In [ ]:
import sys

# Identifier-like literals are interned.
name_a = "hello"
name_b = "hello"

print("simple literals:")
print("   same object?", name_a is name_b)

# Strings built at runtime are usually not.
built_a = "hel" + "lo"          # folded at compile time
built_b = "".join(["h", "e", "l", "l", "o"])   # built at runtime

print("")
print("compile-time concatenation:", built_a is name_a)
print("runtime-built string:      ", built_b is name_a)
print("but equal by value?        ", built_b == name_a)

# Strings with spaces or punctuation are not identifier-like.
spaced_a = "hello world"
spaced_b = "hello world"

print("")
print("with a space (still literals):", spaced_a is spaced_b)

# You can force interning explicitly.
forced_a = sys.intern("".join(["h", "i", "!"]))
forced_b = sys.intern("hi!")

print("")
print("after sys.intern on both:", forced_a is forced_b)

print("")
print("Interning saves memory and speeds up dictionary lookups, since")
print("comparing interned strings can short-circuit on identity.")

## Ids get reused

An id is unique only among **living** objects. Once an object is freed, its id
becomes available again — which produces genuinely startling output.

In [ ]:
# Create a temporary object and record its id.
first_id = id([1, 2, 3])

# That list had no name, so it was freed immediately after id() returned.
second_id = id([4, 5, 6])

print("id of the first temporary list: ", first_id)
print("id of the second temporary list:", second_id)
print("same id?", first_id == second_id)

print("")
print("The lists had different contents and never coexisted. The first")
print("was freed the instant id() returned, so the second could reuse")
print("its address.")

print("")
print("LESSON: an id identifies an object only while it is alive.")
print("Never store an id and compare it later.")

# Keeping a reference prevents reuse.
kept_first = [1, 2, 3]
kept_second = [4, 5, 6]

print("")
print("With references kept, the ids differ:",
      id(kept_first) != id(kept_second))

## `type()` and checking types properly

In [ ]:
samples = [42, 3.14, "text", True, None, [1], (1,), {1}, {"a": 1}, len]

print("Value            type()          isinstance(x, int)")
print("-" * 56)
for value in samples:
    preview = repr(value)
    if len(preview) > 15:
        preview = preview[:12] + "..."
    print(preview.ljust(16), type(value).__name__.ljust(15),
          isinstance(value, int))

print("")
print("Notice True reports isinstance(True, int) as True -")
print("bool is a SUBCLASS of int in Python. Chapter 05 covers this.")

In [ ]:
# type() vs isinstance() - they answer different questions.

class Animal:
    """A base class."""


class Dog(Animal):
    """A subclass."""


pet = Dog()

print("type(pet) is Dog:            ", type(pet) is Dog)
print("type(pet) is Animal:         ", type(pet) is Animal, "<- exact type only")
print("isinstance(pet, Dog):        ", isinstance(pet, Dog))
print("isinstance(pet, Animal):     ", isinstance(pet, Animal), "<- honours inheritance")

print("")
print("RULE: prefer isinstance(). It respects subclasses, which is almost")
print("always what you want. type() is for when you need the EXACT type.")

# isinstance accepts a tuple of types.
value = 3.14
print("")
print("isinstance(3.14, (int, float)):", isinstance(value, (int, float)))

## A practical identity checker

Putting it together into something you can reuse when debugging.

In [ ]:
def describe(label, value):
    """Print the identity, type and value of an object."""
    # ljust keeps the columns aligned regardless of label length.
    print(f"{label:<14} id={id(value):<16} type={type(value).__name__:<8} {value!r}")


# Demonstrate on a shared-object situation.
config = {"debug": True}
alias = config
copy_of_config = dict(config)

print("Two names for one dict, plus a genuine copy:")
describe("config", config)
describe("alias", alias)
describe("copy", copy_of_config)

print("")
print("config is alias:", config is alias)
print("config is copy: ", config is copy_of_config)
print("config == copy: ", config == copy_of_config)

# Changing the original shows which is which.
config["debug"] = False

print("")
print("After config['debug'] = False:")
print("   alias:", alias, "<- shares the object")
print("   copy: ", copy_of_config, "<- independent")

## Takeaways

1. Every object has an **identity**, a **type** (both permanent) and a **value**
   (changeable only if mutable).
2. `id(x)` is the memory address in CPython — an implementation detail, unique
   only among **living** objects.
3. `a is b` is exactly `id(a) == id(b)`. It cannot be overridden.
4. CPython **caches small integers** (−5 to 256) and **interns** identifier-like
   strings, so `is` can be `True` for separate literals.
5. Never use `is` to compare values. Python emits a `SyntaxWarning` if you try.
6. Use `is` only for `None`, `True`, `False`, and genuine identity checks.
7. Prefer `isinstance()` over `type()` — it respects inheritance.
8. Ids are **reused** after an object dies. Never store one for later comparison.

## Try it yourself

1. Check `id()` of the same list before and after `.append()`. Why is it
   unchanged?
2. Find the small-integer cache boundary yourself using `int(str(n))`.
3. Try `x = 1000; y = 1000; x is y` in one cell, then on separate lines. Does the
   answer change? (Compile-time folding is why.)
4. Write a class with `__eq__` returning `True` always. Confirm `is` still works.
5. Use `describe()` above on a list, a copy, and an alias. Which shares identity?